In [ ]:
# =========================================================
# DRFT-LSTM: Dual-Resolution Frequency-Temporal LSTM
# Robust Edge CSI Human Activity Recognition
#
# Single-cell Kaggle code
# Dataset expected:
# /kaggle/input/.../csi-bench/Multitask/HumanActivityRecognition
# =========================================================

import os
import re
import gc
import glob
import json
import h5py
import time
import copy
import random
import shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from IPython.display import FileLink, display

# =========================================================
# CONFIG
# =========================================================

TASK_NAME = "HumanActivityRecognition"

SEED = 42

EPOCHS = 35
PATIENCE = 8
MIN_DELTA = 1e-4

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
ACCUM_STEPS = 4

LR = 8e-4
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
LABEL_SMOOTHING = 0.03

TARGET_SUBCARRIERS = 56
TARGET_TIME_LEN = 500

DROPOUT = 0.1
NUM_WORKERS = 0

# DRFT-LSTM dimensions
FINE_HIDDEN = 96
COARSE_HIDDEN = 64
FREQ_HIDDEN = 64
FUSION_DIM = 96
NUM_FINE_LAYERS = 2
NUM_COARSE_LAYERS = 1
NUM_FREQ_LAYERS = 1
FREQ_BINS = 8
COARSE_DOWNSAMPLE = 4

# Robustness-aware training
ROBUST_TRAINING = True
AUG_PROB = 0.35
AUG_RANDOM_SC_MAX = 0.25
AUG_CONTIG_SC_MAX = 0.20
AUG_NOISE_STD_MAX = 0.05

RUN_ROBUSTNESS_AFTER_TRAIN = True
ROBUSTNESS_TRIALS = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("medium")
else:
    print("WARNING: GPU not detected. Enable GPU in Kaggle notebook settings.")

OUT_DIR = "/kaggle/working/drft_lstm"
os.makedirs(OUT_DIR, exist_ok=True)

BEST_CKPT_PATH = os.path.join(OUT_DIR, "best_drft_lstm.pth")
HISTORY_PATH = os.path.join(OUT_DIR, "drft_lstm_history.csv")
METRICS_PATH = os.path.join(OUT_DIR, "drft_lstm_final_metrics.json")
VAL_REPORT_PATH = os.path.join(OUT_DIR, "drft_lstm_val_report.txt")
TEST_REPORT_PATH = os.path.join(OUT_DIR, "drft_lstm_test_report.txt")
VAL_CM_PATH = os.path.join(OUT_DIR, "drft_lstm_val_confusion_matrix.csv")
TEST_CM_PATH = os.path.join(OUT_DIR, "drft_lstm_test_confusion_matrix.csv")
ROBUSTNESS_PATH = os.path.join(OUT_DIR, "drft_lstm_robustness_results.csv")

# =========================================================
# HELPERS
# =========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

set_seed(SEED)
cleanup()

# =========================================================
# AUTO-DETECT CSI-BENCH ROOT
# =========================================================

def find_task_root(task_name="HumanActivityRecognition"):
    patterns = [
        f"/kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/{task_name}",
        f"/kaggle/input/**/Multitask/{task_name}",
        f"/kaggle/input/**/{task_name}",
    ]

    candidates = []

    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            if os.path.isdir(path):
                candidates.append(path)

    candidates = list(dict.fromkeys(candidates))

    print("\nCandidate task roots:")
    for path in candidates:
        print(" -", path)

    for path in candidates:
        split_file = os.path.join(path, "splits", "train_id.json")
        metadata_file = os.path.join(path, "metadata", "sample_metadata.csv")

        if os.path.exists(split_file) and os.path.exists(metadata_file):
            print("\nUsing ROOT:")
            print(path)
            return path

    raise FileNotFoundError(
        "Could not find HumanActivityRecognition root with splits/train_id.json "
        "and metadata/sample_metadata.csv"
    )

ROOT = find_task_root(TASK_NAME)

METADATA_DIR = os.path.join(ROOT, "metadata")
SPLITS_DIR = os.path.join(ROOT, "splits")
MULTITASK_DIR = os.path.dirname(ROOT)
CSI_BENCH_DIR = os.path.dirname(MULTITASK_DIR)

print("\n========== ROOT VERIFICATION ==========")
print("ROOT:", ROOT)
print("Metadata:", os.path.exists(METADATA_DIR))
print("Splits:", os.path.exists(SPLITS_DIR))
print("sub_Human_h5:", os.path.exists(os.path.join(MULTITASK_DIR, "sub_Human_h5")))

# =========================================================
# SPLIT + LABEL SETUP
# =========================================================

def load_split_ids(root_dir, split):
    split_file = os.path.join(root_dir, "splits", f"{split}_id.json")
    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Missing split file: {split_file}")

    with open(split_file, "r") as f:
        return set(map(str, json.load(f)))

metadata_path = os.path.join(ROOT, "metadata", "sample_metadata.csv")
metadata = pd.read_csv(metadata_path)

metadata["id"] = metadata["id"].astype(str)
metadata["file_path"] = metadata["file_path"].astype(str)

train_ids = load_split_ids(ROOT, "train")
val_ids = load_split_ids(ROOT, "val")
test_ids = load_split_ids(ROOT, "test")

print("\n========== SPLIT CHECK ==========")
print("Train IDs:", len(train_ids))
print("Val IDs  :", len(val_ids))
print("Test IDs :", len(test_ids))
print("Train-Val overlap :", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap  :", len(val_ids & test_ids))

train_meta = metadata[metadata["id"].isin(train_ids)].copy()
train_labels = sorted(train_meta["label"].unique())

LABEL_MAPPING = {label: idx for idx, label in enumerate(train_labels)}
INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}
NUM_CLASSES = len(LABEL_MAPPING)

print("\nLabel mapping:")
print(LABEL_MAPPING)

# =========================================================
# DATASET
# =========================================================

class CSIBenchDataset(Dataset):
    def __init__(
        self,
        root_dir,
        split="train",
        normalize=True,
        target_subcarriers=56,
        target_time_len=500,
        label_mapping=None
    ):
        self.root_dir = root_dir
        self.split = split
        self.normalize = normalize
        self.target_subcarriers = target_subcarriers
        self.target_time_len = target_time_len
        self.label_mapping = label_mapping

        self.metadata_dir = os.path.join(root_dir, "metadata")
        self.splits_dir = os.path.join(root_dir, "splits")
        self.multitask_dir = os.path.dirname(root_dir)
        self.csi_bench_dir = os.path.dirname(self.multitask_dir)

        split_file = os.path.join(self.splits_dir, f"{split}_id.json")
        metadata_file = os.path.join(self.metadata_dir, "sample_metadata.csv")

        if not os.path.exists(split_file):
            raise FileNotFoundError(f"Split file not found: {split_file}")

        if not os.path.exists(metadata_file):
            raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

        with open(split_file, "r") as f:
            self.sample_ids = list(map(str, json.load(f)))

        self.metadata = pd.read_csv(metadata_file)
        self.metadata["id"] = self.metadata["id"].astype(str)
        self.metadata["file_path"] = self.metadata["file_path"].astype(str)

        self.meta_dict = {
            str(row["id"]): row
            for _, row in self.metadata.iterrows()
        }

        print(f"Loaded {len(self.sample_ids)} samples for split: {split}")

    def __len__(self):
        return len(self.sample_ids)

    def resolve_file_path(self, raw_path):
        raw = str(raw_path).replace("\\", "/").strip()

        if raw.startswith("./"):
            raw = raw[2:]

        candidates = []

        if os.path.isabs(raw):
            candidates.append(os.path.normpath(raw))

        candidates.extend([
            os.path.normpath(os.path.join(self.metadata_dir, raw)),
            os.path.normpath(os.path.join(self.root_dir, raw)),
            os.path.normpath(os.path.join(self.multitask_dir, raw)),
            os.path.normpath(os.path.join(self.csi_bench_dir, raw)),
            os.path.normpath(os.path.join("/kaggle/input", raw)),
        ])

        if "sub_Human_h5/" in raw:
            suffix = raw.split("sub_Human_h5/", 1)[1]
            candidates.append(
                os.path.normpath(os.path.join(self.multitask_dir, "sub_Human_h5", suffix))
            )

        if "sub_Human_mat/" in raw:
            suffix = raw.split("sub_Human_mat/", 1)[1]
            candidates.append(
                os.path.normpath(os.path.join(self.multitask_dir, "sub_Human_mat", suffix))
            )

        for path in candidates:
            if os.path.exists(path):
                return path

        base = os.path.basename(raw)
        matches = []

        for search_base in [self.multitask_dir, self.csi_bench_dir]:
            pattern = os.path.join(search_base, "**", base)
            matches.extend(glob.glob(pattern, recursive=True))

        matches = sorted(list(set(matches)))

        if len(matches) == 1:
            return matches[0]

        if len(matches) > 1:
            raise RuntimeError(
                "Ambiguous file resolution. Multiple files share the same basename.\n"
                f"metadata file_path: {raw_path}\n"
                f"basename: {base}\n"
                "Matches:\n" + "\n".join(matches[:20])
            )

        raise FileNotFoundError(
            "Could not resolve CSI file path.\n"
            f"metadata file_path: {raw_path}\n"
            f"basename searched: {base}"
        )

    def load_h5(self, path):
        with h5py.File(path, "r") as f:
            keys = list(f.keys())

            for key in ["csi", "data", "CSI", "amplitude"]:
                if key in keys:
                    return f[key][:]

            return f[keys[0]][:]

    def to_ckt(self, data):
        data = np.array(data)

        if np.iscomplexobj(data):
            data = np.abs(data)

        data = data.astype(np.float32)

        if data.ndim == 2:
            a, b = data.shape

            if a <= b:
                return data[np.newaxis, :, :]
            else:
                return data.T[np.newaxis, :, :]

        if data.ndim == 3:
            s0, s1, s2 = data.shape

            if s0 <= 8 and s1 <= self.target_subcarriers * 2:
                return data

            if s2 <= 8 and s0 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 0, 1))

            if s2 <= 8 and s1 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 1, 0))

            if s2 <= 16:
                return np.transpose(data, (2, 0, 1))

        raise ValueError(f"Unexpected CSI shape: {data.shape}")

    def standardize_subcarriers(self, x):
        C, K, T = x.shape

        if K > self.target_subcarriers:
            x = x[:, :self.target_subcarriers, :]
        elif K < self.target_subcarriers:
            pad = np.zeros((C, self.target_subcarriers - K, T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=1)

        return x

    def standardize_time(self, x):
        C, K, T = x.shape

        if T > self.target_time_len:
            x = x[:, :, :self.target_time_len]
        elif T < self.target_time_len:
            pad = np.zeros((C, K, self.target_time_len - T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=2)

        return x

    def __getitem__(self, idx):
        sample_id = str(self.sample_ids[idx])

        if sample_id not in self.meta_dict:
            raise KeyError(f"Sample ID not found in metadata: {sample_id}")

        meta = self.meta_dict[sample_id]

        file_path = self.resolve_file_path(meta["file_path"])
        csi = self.load_h5(file_path)

        x = self.to_ckt(csi)
        x = self.standardize_subcarriers(x)
        x = self.standardize_time(x)

        if self.normalize:
            x = (x - x.mean()) / (x.std() + 1e-6)

        label_name = meta["label"]

        if label_name not in self.label_mapping:
            raise KeyError(f"Label not in mapping: {label_name}")

        y = self.label_mapping[label_name]

        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# =========================================================
# CREATE DATASETS + LOADERS
# =========================================================

train_dataset = CSIBenchDataset(
    ROOT,
    split="train",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING
)

val_dataset = CSIBenchDataset(
    ROOT,
    split="val",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING
)

test_dataset = CSIBenchDataset(
    ROOT,
    split="test",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING
)

sample_x, sample_y = train_dataset[0]
CSI_CHANNELS = sample_x.shape[0]

print("\n========== DATA INFO ==========")
print("Sample shape:", sample_x.shape)
print("CSI channels:", CSI_CHANNELS)
print("Num classes:", NUM_CLASSES)
print("Example label:", sample_y)

def make_loaders(seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda"
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda"
    )

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(SEED)

cleanup()

# =========================================================
# ROBUST TRAINING AUGMENTATION
# =========================================================

def random_subcarrier_mask_batch(x, drop_prob):
    B, C, K, T = x.shape
    mask = (torch.rand(B, 1, K, 1, device=x.device) > drop_prob).float()
    return x * mask

def contiguous_subcarrier_mask_batch(x, drop_ratio):
    B, C, K, T = x.shape
    width = max(1, int(K * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, K - width + 1, (1,), device=x.device).item()
        out[b, :, start:start + width, :] = 0.0

    return out

def apply_train_augmentation(x):
    if not ROBUST_TRAINING:
        return x

    if torch.rand(1).item() > AUG_PROB:
        return x

    aug_choice = torch.rand(1).item()

    if aug_choice < 0.40:
        drop_prob = float(np.random.uniform(0.05, AUG_RANDOM_SC_MAX))
        x = random_subcarrier_mask_batch(x, drop_prob)

    elif aug_choice < 0.75:
        drop_ratio = float(np.random.uniform(0.05, AUG_CONTIG_SC_MAX))
        x = contiguous_subcarrier_mask_batch(x, drop_ratio)

    else:
        std = float(np.random.uniform(0.01, AUG_NOISE_STD_MAX))
        x = x + torch.randn_like(x) * std

    return x

# =========================================================
# MODEL: DRFT-LSTM
# =========================================================

class AttentionPool1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        # x: (B, T, D)
        weights = torch.softmax(self.score(x), dim=1)
        return torch.sum(weights * x, dim=1)


class FrequencyFeatureEncoder(nn.Module):
    """
    Per-time-step frequency/subcarrier feature extraction.
    Converts (B, C, K, T) into (B, T, FREQ_HIDDEN).
    """
    def __init__(self, csi_channels, freq_hidden=64, freq_bins=8, dropout=0.1):
        super().__init__()

        self.freq_bins = freq_bins

        self.freq_conv = nn.Sequential(
            nn.Conv1d(csi_channels, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(16),
            nn.GELU(),

            nn.Conv1d(16, 16, kernel_size=5, padding=2, groups=16, bias=False),
            nn.Conv1d(16, 32, kernel_size=1, bias=False),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(freq_bins)
        )

        self.proj = nn.Sequential(
            nn.LayerNorm(32 * freq_bins),
            nn.Linear(32 * freq_bins, freq_hidden),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        B, C, K, T = x.shape

        # (B, C, K, T) -> (B*T, C, K)
        z = x.permute(0, 3, 1, 2).contiguous()
        z = z.reshape(B * T, C, K)

        z = self.freq_conv(z)
        z = z.reshape(B * T, -1)
        z = self.proj(z)

        z = z.reshape(B, T, -1)

        return z


class DRFTLSTM(nn.Module):
    def __init__(
        self,
        num_classes,
        csi_channels,
        num_subcarriers,
        fine_hidden=96,
        coarse_hidden=64,
        freq_hidden=64,
        fusion_dim=96,
        dropout=0.1
    ):
        super().__init__()

        self.input_dim = csi_channels * num_subcarriers
        self.coarse_downsample = COARSE_DOWNSAMPLE

        # Fine temporal branch: full 500-step temporal sequence
        self.fine_input_proj = nn.Sequential(
            nn.LayerNorm(self.input_dim),
            nn.Linear(self.input_dim, fine_hidden),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.fine_lstm = nn.LSTM(
            input_size=fine_hidden,
            hidden_size=fine_hidden,
            num_layers=NUM_FINE_LAYERS,
            batch_first=True,
            dropout=dropout if NUM_FINE_LAYERS > 1 else 0.0,
            bidirectional=False
        )

        self.fine_pool = AttentionPool1D(fine_hidden)

        # Coarse temporal branch: downsampled long-range branch
        self.coarse_input_proj = nn.Sequential(
            nn.LayerNorm(self.input_dim),
            nn.Linear(self.input_dim, coarse_hidden),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.coarse_lstm = nn.LSTM(
            input_size=coarse_hidden,
            hidden_size=coarse_hidden,
            num_layers=NUM_COARSE_LAYERS,
            batch_first=True,
            dropout=dropout if NUM_COARSE_LAYERS > 1 else 0.0,
            bidirectional=False
        )

        self.coarse_pool = AttentionPool1D(coarse_hidden)

        # Frequency/subcarrier-focused branch
        self.freq_encoder = FrequencyFeatureEncoder(
            csi_channels=csi_channels,
            freq_hidden=freq_hidden,
            freq_bins=FREQ_BINS,
            dropout=dropout
        )

        self.freq_lstm = nn.LSTM(
            input_size=freq_hidden,
            hidden_size=freq_hidden,
            num_layers=NUM_FREQ_LAYERS,
            batch_first=True,
            dropout=dropout if NUM_FREQ_LAYERS > 1 else 0.0,
            bidirectional=False
        )

        self.freq_pool = AttentionPool1D(freq_hidden)

        # Project all branches into a common fusion space
        self.fine_to_fusion = nn.Linear(fine_hidden, fusion_dim)
        self.coarse_to_fusion = nn.Linear(coarse_hidden, fusion_dim)
        self.freq_to_fusion = nn.Linear(freq_hidden, fusion_dim)

        # Soft adaptive branch weighting
        self.branch_gate = nn.Sequential(
            nn.LayerNorm(fusion_dim * 3),
            nn.Linear(fusion_dim * 3, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, 3)
        )

        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes)
        )

    def forward(self, x):
        B, C, K, T = x.shape

        # Fine temporal sequence
        x_flat = x.reshape(B, C * K, T).transpose(1, 2)  # (B, T, C*K)
        fine_in = self.fine_input_proj(x_flat)
        fine_out, _ = self.fine_lstm(fine_in)
        fine_vec = self.fine_pool(fine_out)

        # Coarse temporal sequence
        x_coarse = x.reshape(B, C * K, T)
        x_coarse = F.avg_pool1d(
            x_coarse,
            kernel_size=self.coarse_downsample,
            stride=self.coarse_downsample
        )
        x_coarse = x_coarse.transpose(1, 2)  # (B, T/4, C*K)

        coarse_in = self.coarse_input_proj(x_coarse)
        coarse_out, _ = self.coarse_lstm(coarse_in)
        coarse_vec = self.coarse_pool(coarse_out)

        # Frequency-focused sequence
        freq_in = self.freq_encoder(x)
        freq_out, _ = self.freq_lstm(freq_in)
        freq_vec = self.freq_pool(freq_out)

        # Common fusion space
        fine_f = self.fine_to_fusion(fine_vec)
        coarse_f = self.coarse_to_fusion(coarse_vec)
        freq_f = self.freq_to_fusion(freq_vec)

        concat = torch.cat([fine_f, coarse_f, freq_f], dim=-1)
        gate_logits = self.branch_gate(concat)
        gate_weights = torch.softmax(gate_logits, dim=-1)

        fused = (
            gate_weights[:, 0:1] * fine_f +
            gate_weights[:, 1:2] * coarse_f +
            gate_weights[:, 2:3] * freq_f
        )

        logits = self.head(fused)

        return logits

# =========================================================
# INITIALIZE MODEL
# =========================================================

model = DRFTLSTM(
    num_classes=NUM_CLASSES,
    csi_channels=CSI_CHANNELS,
    num_subcarriers=TARGET_SUBCARRIERS,
    fine_hidden=FINE_HIDDEN,
    coarse_hidden=COARSE_HIDDEN,
    freq_hidden=FREQ_HIDDEN,
    fusion_dim=FUSION_DIM,
    dropout=DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n========== DRFT-LSTM MODEL ==========")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

cleanup()

# =========================================================
# TRAIN / EVAL FUNCTIONS
# =========================================================

def train_one_epoch(epoch):
    model.train()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(train_loader, desc=f"Epoch {epoch} Train", leave=False)

    for step, (x, y) in enumerate(progress):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        x = apply_train_augmentation(x)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(x)
            loss = criterion(logits, y)
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * ACCUM_STEPS

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        progress.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(train_loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0)
    }


@torch.no_grad()
def evaluate_loader(loader, split_name="Val"):
    model.eval()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    progress = tqdm(loader, desc=split_name, leave=False)

    for x, y in progress:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(x)
            loss = criterion(logits, y)

        running_loss += loss.item()

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
        "labels": labels_all,
        "preds": preds_all
    }

# =========================================================
# TRAINING LOOP
# =========================================================

best_val_macro_f1 = 0.0
best_val_acc = 0.0
best_val_weighted_f1 = 0.0
best_epoch = 0
patience_counter = 0
history = []

print("\nStarting DRFT-LSTM training...\n")

for epoch in range(1, EPOCHS + 1):
    cleanup()

    train_metrics = train_one_epoch(epoch)
    val_metrics = evaluate_loader(val_loader, split_name=f"Epoch {epoch} Val")

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "train_weighted_f1": train_metrics["weighted_f1"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
    }

    history.append(row)

    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] | "
        f"Train Acc: {row['train_acc']*100:.2f}% | "
        f"Train Macro-F1: {row['train_macro_f1']*100:.2f}% | "
        f"Val Acc: {row['val_acc']*100:.2f}% | "
        f"Val Macro-F1: {row['val_macro_f1']*100:.2f}% | "
        f"Val Weighted-F1: {row['val_weighted_f1']*100:.2f}%"
    )

    improved = row["val_macro_f1"] > best_val_macro_f1 + MIN_DELTA

    if improved:
        best_val_macro_f1 = row["val_macro_f1"]
        best_val_acc = row["val_acc"]
        best_val_weighted_f1 = row["val_weighted_f1"]
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": "DRFTLSTM",
                "seed": SEED,
                "best_epoch": best_epoch,
                "best_val_acc": best_val_acc,
                "best_val_macro_f1": best_val_macro_f1,
                "best_val_weighted_f1": best_val_weighted_f1,
                "label_mapping": LABEL_MAPPING,
                "config": {
                    "target_subcarriers": TARGET_SUBCARRIERS,
                    "target_time_len": TARGET_TIME_LEN,
                    "csi_channels": CSI_CHANNELS,
                    "num_classes": NUM_CLASSES,
                    "fine_hidden": FINE_HIDDEN,
                    "coarse_hidden": COARSE_HIDDEN,
                    "freq_hidden": FREQ_HIDDEN,
                    "fusion_dim": FUSION_DIM,
                    "robust_training": ROBUST_TRAINING,
                    "aug_prob": AUG_PROB
                },
                "history": history
            },
            BEST_CKPT_PATH
        )

        print(
            f"Saved best DRFT-LSTM | "
            f"Epoch {best_epoch} | "
            f"Val Acc {best_val_acc*100:.2f}% | "
            f"Val Macro-F1 {best_val_macro_f1*100:.2f}% | "
            f"Val Weighted-F1 {best_val_weighted_f1*100:.2f}%"
        )

    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}")
        break

pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

print("\nTraining complete.")
print(f"Best Epoch           : {best_epoch}")
print(f"Best Val Acc         : {best_val_acc*100:.2f}%")
print(f"Best Val Macro-F1    : {best_val_macro_f1*100:.2f}%")
print(f"Best Val Weighted-F1 : {best_val_weighted_f1*100:.2f}%")

# =========================================================
# FINAL EVALUATION
# =========================================================

print("\n========== LOADING BEST CHECKPOINT ==========")

ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Loaded best epoch:", ckpt["best_epoch"])

target_names = [INV_LABEL_MAPPING[i] for i in range(NUM_CLASSES)]

val_final = evaluate_loader(val_loader, split_name="Final Val")
test_final = evaluate_loader(test_loader, split_name="Final Test")

val_report = classification_report(
    val_final["labels"],
    val_final["preds"],
    target_names=target_names,
    digits=4,
    zero_division=0
)

test_report = classification_report(
    test_final["labels"],
    test_final["preds"],
    target_names=target_names,
    digits=4,
    zero_division=0
)

val_cm = confusion_matrix(val_final["labels"], val_final["preds"])
test_cm = confusion_matrix(test_final["labels"], test_final["preds"])

with open(VAL_REPORT_PATH, "w") as f:
    f.write(val_report)

with open(TEST_REPORT_PATH, "w") as f:
    f.write(test_report)

pd.DataFrame(val_cm, index=target_names, columns=target_names).to_csv(VAL_CM_PATH)
pd.DataFrame(test_cm, index=target_names, columns=target_names).to_csv(TEST_CM_PATH)

print("\n========== FINAL VALIDATION METRICS ==========")
print(f"Val Loss        : {val_final['loss']:.4f}")
print(f"Val Accuracy    : {val_final['accuracy']*100:.2f}%")
print(f"Val Macro-F1    : {val_final['macro_f1']*100:.2f}%")
print(f"Val Weighted-F1 : {val_final['weighted_f1']*100:.2f}%")

print("\n========== FINAL TEST METRICS ==========")
print(f"Test Loss        : {test_final['loss']:.4f}")
print(f"Test Accuracy    : {test_final['accuracy']*100:.2f}%")
print(f"Test Macro-F1    : {test_final['macro_f1']*100:.2f}%")
print(f"Test Weighted-F1 : {test_final['weighted_f1']*100:.2f}%")

print("\n========== TEST CLASSIFICATION REPORT ==========")
print(test_report)

print("\n========== TEST CONFUSION MATRIX ==========")
print(test_cm)

# =========================================================
# EDGE PROFILE
# =========================================================

def model_size_mb(model):
    temp_path = os.path.join(OUT_DIR, "temp_model_size.pth")
    torch.save(model.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 ** 2)
    os.remove(temp_path)
    return size_mb

@torch.no_grad()
def profile_latency(model_obj, device, input_shape, warmup=20, runs=50):
    model_obj.eval()

    dummy = torch.randn(*input_shape).to(device)

    for _ in range(warmup):
        _ = model_obj(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    for _ in range(runs):
        _ = model_obj(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    latency_ms = ((end - start) / runs) * 1000

    peak_mem_mb = None
    if device.type == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return latency_ms, peak_mem_mb

print("\n========== EDGE PROFILE ==========")

edge_input_shape = (1, CSI_CHANNELS, TARGET_SUBCARRIERS, TARGET_TIME_LEN)

final_params = sum(p.numel() for p in model.parameters())
final_size_mb = model_size_mb(model)

cuda_latency_ms, peak_mem_mb = profile_latency(
    model,
    DEVICE,
    edge_input_shape,
    warmup=20,
    runs=50
)

model_cpu = copy.deepcopy(model).cpu().eval()

cpu_latency_ms, _ = profile_latency(
    model_cpu,
    torch.device("cpu"),
    edge_input_shape,
    warmup=10,
    runs=30
)

print(f"Input shape           : {edge_input_shape}")
print(f"Total parameters     : {final_params:,}")
print(f"Model size           : {final_size_mb:.3f} MB")
print(f"CUDA latency         : {cuda_latency_ms:.3f} ms/sample")
print(f"CPU latency          : {cpu_latency_ms:.3f} ms/sample")

if peak_mem_mb is not None:
    print(f"Peak inference memory: {peak_mem_mb:.2f} MB")

# =========================================================
# ROBUSTNESS EVALUATION
# =========================================================

def perturb_clean(x):
    return x

def perturb_gaussian_noise(x, std=0.2):
    return x + torch.randn_like(x) * std

def perturb_random_subcarrier_mask(x, drop_prob=0.3):
    B, C, K, T = x.shape
    mask = (torch.rand(B, 1, K, 1, device=x.device) > drop_prob).float()
    return x * mask

def perturb_contiguous_subcarrier_mask(x, drop_ratio=0.3):
    B, C, K, T = x.shape
    width = max(1, int(K * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, K - width + 1, (1,), device=x.device).item()
        out[b, :, start:start + width, :] = 0.0

    return out

def perturb_temporal_mask(x, drop_ratio=0.3):
    B, C, K, T = x.shape
    width = max(1, int(T * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, T - width + 1, (1,), device=x.device).item()
        out[b, :, :, start:start + width] = 0.0

    return out

def perturb_time_shift(x, shift=50):
    return torch.roll(x, shifts=shift, dims=-1)

def perturb_crop_resize(x, keep_ratio=0.75):
    B, C, K, T = x.shape
    keep_len = max(8, int(T * keep_ratio))
    out_list = []

    for b in range(B):
        start = torch.randint(0, T - keep_len + 1, (1,), device=x.device).item()
        crop = x[b:b+1, :, :, start:start + keep_len]
        crop = crop.reshape(1, C * K, keep_len)

        resized = F.interpolate(
            crop,
            size=T,
            mode="linear",
            align_corners=False
        )

        resized = resized.reshape(1, C, K, T)
        out_list.append(resized)

    return torch.cat(out_list, dim=0)

def perturb_combined_harsh(x):
    x = perturb_gaussian_noise(x, std=0.20)
    x = perturb_random_subcarrier_mask(x, drop_prob=0.40)
    x = perturb_temporal_mask(x, drop_ratio=0.20)
    return x

ROBUSTNESS_CONDITIONS = [
    {"name": "clean", "fn": perturb_clean, "kwargs": {}, "trials": 1},
    {"name": "gaussian_noise_0.20", "fn": perturb_gaussian_noise, "kwargs": {"std": 0.20}, "trials": ROBUSTNESS_TRIALS},
    {"name": "random_subcarrier_mask_30", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "random_subcarrier_mask_50", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.50}, "trials": ROBUSTNESS_TRIALS},
    {"name": "contiguous_subcarrier_mask_30", "fn": perturb_contiguous_subcarrier_mask, "kwargs": {"drop_ratio": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "temporal_mask_30", "fn": perturb_temporal_mask, "kwargs": {"drop_ratio": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "time_shift_50", "fn": perturb_time_shift, "kwargs": {"shift": 50}, "trials": 1},
    {"name": "crop_resize_75", "fn": perturb_crop_resize, "kwargs": {"keep_ratio": 0.75}, "trials": ROBUSTNESS_TRIALS},
    {"name": "combined_harsh", "fn": perturb_combined_harsh, "kwargs": {}, "trials": ROBUSTNESS_TRIALS},
]

@torch.no_grad()
def evaluate_under_condition(model_obj, loader, condition, trial_seed=42):
    set_seed(trial_seed)

    model_obj.eval()

    preds_all = []
    labels_all = []

    fn = condition["fn"]
    kwargs = condition["kwargs"]

    for x, y in tqdm(loader, desc=condition["name"], leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        x = fn(x, **kwargs)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model_obj(x)

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, preds

    return {
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0)
    }

if RUN_ROBUSTNESS_AFTER_TRAIN:
    print("\n========== ROBUSTNESS EVALUATION ==========")

    robustness_rows = []

    for condition in ROBUSTNESS_CONDITIONS:
        trial_results = []

        for t in range(condition["trials"]):
            trial_seed = SEED + 1000 * t
            metrics = evaluate_under_condition(
                model,
                test_loader,
                condition,
                trial_seed=trial_seed
            )
            trial_results.append(metrics)

        accs = np.array([r["accuracy"] for r in trial_results])
        macros = np.array([r["macro_f1"] for r in trial_results])
        weighteds = np.array([r["weighted_f1"] for r in trial_results])

        row = {
            "condition": condition["name"],
            "trials": condition["trials"],
            "acc_mean": float(accs.mean()),
            "acc_std": float(accs.std(ddof=1)) if len(accs) > 1 else 0.0,
            "macro_f1_mean": float(macros.mean()),
            "macro_f1_std": float(macros.std(ddof=1)) if len(macros) > 1 else 0.0,
            "weighted_f1_mean": float(weighteds.mean()),
            "weighted_f1_std": float(weighteds.std(ddof=1)) if len(weighteds) > 1 else 0.0,
        }

        robustness_rows.append(row)

        print(
            f"{row['condition']:35s} | "
            f"Acc {row['acc_mean']*100:.2f} | "
            f"Macro-F1 {row['macro_f1_mean']*100:.2f} | "
            f"Weighted-F1 {row['weighted_f1_mean']*100:.2f}"
        )

    robustness_df = pd.DataFrame(robustness_rows)

    clean_weighted = robustness_df[robustness_df["condition"] == "clean"]["weighted_f1_mean"].iloc[0]
    robustness_df["weighted_f1_drop"] = clean_weighted - robustness_df["weighted_f1_mean"]

    percent_df = robustness_df.copy()
    for col in ["acc_mean", "acc_std", "macro_f1_mean", "macro_f1_std", "weighted_f1_mean", "weighted_f1_std", "weighted_f1_drop"]:
        percent_df[col] = percent_df[col] * 100

    percent_df.to_csv(ROBUSTNESS_PATH, index=False)

    print("\n========== ROBUSTNESS RESULTS (%) ==========")
    display(percent_df)

# =========================================================
# SAVE FINAL METRICS
# =========================================================

final_metrics = {
    "model": "DRFTLSTM",
    "seed": SEED,
    "best_epoch": int(best_epoch),
    "params": int(final_params),
    "model_size_mb": float(final_size_mb),
    "cuda_latency_ms": float(cuda_latency_ms),
    "cpu_latency_ms": float(cpu_latency_ms),
    "peak_mem_mb": float(peak_mem_mb) if peak_mem_mb is not None else None,
    "val": {
        "loss": float(val_final["loss"]),
        "accuracy": float(val_final["accuracy"]),
        "macro_f1": float(val_final["macro_f1"]),
        "weighted_f1": float(val_final["weighted_f1"])
    },
    "test": {
        "loss": float(test_final["loss"]),
        "accuracy": float(test_final["accuracy"]),
        "macro_f1": float(test_final["macro_f1"]),
        "weighted_f1": float(test_final["weighted_f1"])
    },
    "label_mapping": LABEL_MAPPING
}

with open(METRICS_PATH, "w") as f:
    json.dump(final_metrics, f, indent=4)

# =========================================================
# TORCHSCRIPT EXPORT
# =========================================================

TS_PATH = os.path.join(OUT_DIR, "drft_lstm_torchscript.pt")

try:
    model_cpu_export = copy.deepcopy(model).cpu().eval()
    example_cpu = torch.randn(*edge_input_shape).cpu()

    with torch.inference_mode():
        traced = torch.jit.trace(
            model_cpu_export,
            example_cpu,
            check_trace=False
        )

    traced.save(TS_PATH)

    print("\nTorchScript export successful.")
    print("TorchScript path:", TS_PATH)
    print(f"TorchScript size: {os.path.getsize(TS_PATH) / (1024 ** 2):.3f} MB")

except Exception as e:
    print("\nTorchScript export failed.")
    print(e)

# =========================================================
# DOWNLOAD ARTIFACTS
# =========================================================

ARTIFACT_ZIP_PATH = "/kaggle/working/drft_lstm_artifacts.zip"

if os.path.exists(ARTIFACT_ZIP_PATH):
    os.remove(ARTIFACT_ZIP_PATH)

shutil.make_archive(
    base_name=ARTIFACT_ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=OUT_DIR
)

print("\n========== SAVED FILES ==========")
print(BEST_CKPT_PATH)
print(HISTORY_PATH)
print(METRICS_PATH)
print(TEST_REPORT_PATH)
print(TEST_CM_PATH)
print(ROBUSTNESS_PATH if RUN_ROBUSTNESS_AFTER_TRAIN else "Robustness skipped")
print(TS_PATH if os.path.exists(TS_PATH) else "TorchScript not available")

print("\n========== DOWNLOAD LINKS ==========")
display(FileLink(ARTIFACT_ZIP_PATH))
display(FileLink(BEST_CKPT_PATH))

if os.path.exists(TS_PATH):
    display(FileLink(TS_PATH))

cleanup()